# Mix NaVILA replay data into the deictic fine-tuning set

Fine-tuning NaVILA only on our pointing episodes risks catastrophic forgetting: the model can lose the
forward / stop / route-following behaviour it learned from its original data. The fix is replay: mix
a sample of NaVILA's own training data into every epoch.

What NaVILA released ([a8cheng/NaVILA-Dataset](https://huggingface.co/datasets/a8cheng/NaVILA-Dataset)):
R2R and RxR (annotations + frames) are usable as replay. Human (YouTube) has annotations only (frames
need `yt-dlp`), EnvDrop needs Habitat rendering, the general VQA sets are not in the repo. Replay here
is **R2R + RxR**. Their frames are Matterport3D renders: the Matterport3D terms of use apply.

1. Download NaVILA's annotations and check the format our export must match
2. Pick replay trajectories from R2R / RxR (and held-out ones to measure forgetting)
3. Fetch only those trajectories' frames (streams the 20 + 24 GB tars once, keeps ~2 GB)
4. Convert our episodes (`make collect` sessions) to the same format
5. Mix: ratio, registration snippet and LoRA script for NaVILA, checks, samples as the model sees them

Setup: `pip install -r data_pipeline/export/requirements.txt`, run from anywhere inside the repo.
Everything goes under `DATA_DIR` (default `data/`, gitignored, same variable as the Makefile):
`navila/` (downloads), `navila_mix/` (what training reads), and our sessions in `sessions/`.
Copy `navila/` and `navila_mix/` to the training machine (or run this there) and set `TRAIN_DATA_DIR`
to where they will be.

In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_pipeline").is_dir())
sys.path.insert(0, str(ROOT))

from data_pipeline.export import mix
from data_pipeline.export.deictic_export import export_sessions
from data_pipeline.export.navila_data import (FRAME_DIRS, download_annotations, fetch_frames, frame_step_check,
                                              parse_answer, pick_trajectories, scan, stop_share, summary,
                                              write_subsets)
from data_pipeline.real.navila_input import load_params
from data_pipeline.real.odom_actions import (FORWARD, FORWARD_UNIT_M, MAX_FORWARD_M, MAX_TURN_DEG, STOP,
                                             TURN_LEFT, TURN_RIGHT, TURN_UNIT_DEG, Action)

DATA = Path(os.environ.get("DATA_DIR", ROOT / "data")).resolve()
NAVILA_DIR = DATA / "navila"          # NaVILA downloads: annotations, frames of the picked trajectories
MIX_DIR = DATA / "navila_mix"         # what training reads (with NAVILA_DIR's frames)
SESSIONS_DIR = DATA / "sessions"      # make collect output
TRAIN_DATA_DIR = os.environ.get("TRAIN_DATA_DIR", str(DATA))  # DATA_DIR as seen on the training machine

SOURCES = ["r2r", "rxr"]              # replay sets
REPLAY = {"r2r": 400, "rxr": 200}     # trajectories mixed into training (~13k + ~12.5k records, ~1.5 GB of frames)
EVAL = {"r2r": 50, "rxr": 50}         # other trajectories, to measure forgetting
REPLAY_PER_DEICTIC = 2.0              # replay records per deictic record in an epoch (1:2); our set is repeated to get there
EVAL_FRACTION = 0.15                  # of our episodes, held out
STRICT = True                         # leave out episodes with any problem (see step 4)
FETCH_FRAMES = os.environ.get("FETCH_FRAMES", "1") == "1"  # step 3 downloads 44 GB: 0 to skip it
SEED = 0
print(f"DATA_DIR {DATA}\nTRAIN_DATA_DIR {TRAIN_DATA_DIR}")

## 1. NaVILA's annotations

About 1.8 GB for R2R + RxR (317 MB + 1.46 GB). Files already in `NAVILA_DIR` are not downloaded again.

In [ ]:
paths = download_annotations(SOURCES, NAVILA_DIR)
for name, p in paths.items():
    print(f"{name:6} {p}  {p.stat().st_size / 1e6:,.0f} MB")

### Scan

One streaming pass per file (about 5 s for R2R, under a minute for RxR, under 1 GB RAM). Each record is
one decision step; `video_id` is `<trajectory>-<step>`; some steps are repeated to balance labels.

In [ ]:
scans = {name: scan(name, p, progress_every=200_000) for name, p in paths.items()}
stats = {name: summary(s) for name, s in scans.items()}
for key in stats[SOURCES[0]]:
    print(f"{key:40}" + "".join(f"  {name}: {stats[name][key]}" for name in stats))

### Check: our action wording is NaVILA's, word for word

Every released answer must parse, and `Action.answer()` (what our export writes) must reproduce it
exactly. Our labeller can emit 25/50/75 cm, 15/30/45 degree and stop: those 10 answers must be the
released ones, no more, no less.

In [ ]:
released = set().union(*(s.answers for s in scans.values()))
for s in scans.values():
    assert not s.unknown, f"{s.name}: answers our parser does not know: {s.unknown.most_common(5)}"
assert all(parse_answer(a).answer() == a for a in released), "an answer does not round-trip"

ours = {Action(STOP, 0, 0).answer()}
for k in range(1, round(MAX_FORWARD_M / FORWARD_UNIT_M) + 1):
    ours.add(Action(FORWARD, k * FORWARD_UNIT_M, 0).answer())
for kind in (TURN_LEFT, TURN_RIGHT):
    for k in range(1, round(MAX_TURN_DEG / TURN_UNIT_DEG) + 1):
        ours.add(Action(kind, k * TURN_UNIT_DEG, 0).answer())
assert ours == released, f"only ours: {ours - released}, only released: {released - ours}"
print(f"OK: the {len(released)} released answers are exactly what Action.answer() writes")
for a in sorted(released):
    print("  ", a)

### Check: one history frame per 25 cm / 15 degree unit

Between consecutive steps, how many frames does the history grow by? If it grows by the number of
25 cm / 15 degree units in the action, our export must add one frame per unit (as `navila_node` already
does at inference), not one per action.

In [ ]:
for name, s in scans.items():
    c = frame_step_check(s)
    total = sum(c.values())
    print(name, {k: f"{v:,} ({v / total:.1%})" for k, v in c.most_common()})
    assert c["per action"] == 0 and c["other"] == 0, f"{name}: history does not grow one frame per unit"
print("OK: one frame per unit")

### Stop balancing

NaVILA repeats each trajectory's stop record (3x R2R, 5x RxR). What our export copies is the resulting
**share** of stop records, not the repeat count: NaVILA's trajectories are 25-40 steps, ours a few, so a
5x repeat would make stops several times more common in our data than NaVILA ever saw.

In [ ]:
for name, st in stats.items():
    print(f"{name}: stop repeated {st['stop record copies -> trajectories']}, "
          f"other steps {st['move record copies -> steps']}, stops = {stop_share(scans[name]):.1%} of records")
TARGET_STOP_SHARE = sum(stop_share(scans[n]) * scans[n].records for n in SOURCES) / sum(scans[n].records for n in SOURCES)
print(f"TARGET_STOP_SHARE = {TARGET_STOP_SHARE:.3f}")

## 2. Replay and held-out trajectories

Whole trajectories, at random (seeded): all their steps and repeats, so NaVILA's label balance comes
along. `*_eval` trajectories are never trained on here, but the base model saw them in its own training:
they measure what fine-tuning **loses** (action accuracy before vs after), not generalisation. For that,
VLN-CE val-unseen in Habitat is the real test.

In [ ]:
subsets, needed = {}, {}
for name in SOURCES:
    replay, held_out = pick_trajectories(scans[name], REPLAY[name], EVAL[name], SEED)
    out = {"replay": MIX_DIR / f"{name}_replay.json", "eval": MIX_DIR / f"{name}_eval.json"}
    res = write_subsets(paths[name], {"replay": replay, "eval": held_out}, out)
    needed[name] = res["replay"]["frames"] | res["eval"]["frames"]
    for g in out:
        subsets[f"{name}_{g}"] = {"data": out[g], "images": NAVILA_DIR / FRAME_DIRS[name], "records": res[g]["records"]}
    (MIX_DIR / f"{name}_frames.json").write_text(json.dumps(sorted(needed[name])))
    print(f"{name}: {len(replay)} replay trajectories -> {res['replay']['records']:,} records, "
          f"{len(held_out)} eval -> {res['eval']['records']:,}; {len(needed[name]):,} frames needed")
del scans  # free the scan memory before the frames

## 3. Frames of those trajectories

The tars (R2R 20 GB, RxR 24 GB) have no index: each is streamed once from Hugging Face and only the
needed frames are written. Expect roughly network speed: ~15 min at 50 MB/s, over an hour at 10 MB/s.
Frames already on disk are skipped, so this can be re-run after an interruption; dropped connections
are retried. `FETCH_FRAMES=0` skips it (step 5 then reports the replay sets as not ready).

In [ ]:
if FETCH_FRAMES:
    for name in SOURCES:
        print(name, fetch_frames(name, needed[name], NAVILA_DIR))
else:
    print("FETCH_FRAMES=0: frames not fetched")

## 4. Our episodes

Every kept episode in `sessions/session_*/ep_*` (discarded ones are in `discarded/` and not read). The
history of each decision is what `navila_node` gives NaVILA at inference: the pointing frames before the
instruction, the frame at the second Enter, then one frame per 25 cm / 15 degree unit (when odometry
says it was completed). Settings (`image_mode`, `gesture_frames`, `gesture_window_s`) come from
`configs/robots/roxy.yaml`, the same file inference reads: change them only with a re-export and re-train.

With `STRICT`, an episode with any problem is left out: strafing / reversing (labels wrong), missing
pointing frames (point for `gesture_window_s` before the second Enter), camera gaps.
`make preview` shows the same problems per session.

In [ ]:
params = load_params()
print("inference settings:", params)
deictic = export_sessions(SESSIONS_DIR, MIX_DIR / "deictic", params, TARGET_STOP_SHARE,
                          eval_fraction=EVAL_FRACTION, seed=SEED, strict=STRICT)
for e in deictic.pop("per episode"):
    flag = "" if not e["problems"] else "  ! " + "; ".join(e["problems"])
    print(f"{e['uid']:40} {e['split']:9} {e['steps']:3} steps, {e['pointing frames']} pointing{flag}")
print(json.dumps(deictic, indent=1))
for split in ("train", "eval"):
    subsets["deictic" if split == "train" else "deictic_eval"] = {
        "data": MIX_DIR / "deictic" / f"{split}.json", "images": MIX_DIR / "deictic" / "frames",
        "records": deictic[split]["records"]}

## 5. Mix

The ratio is set with NaVILA's own mechanism: `--data_mixture deictic*K+r2r_replay+rxr_replay` repeats
our set K times per epoch so there are about `REPLAY_PER_DEICTIC` replay records per deictic record.
Large K means our few episodes are seen many times: watch for overfitting (eval loss on `deictic_eval`).
If K gets large, collect more episodes or lower `REPLAY` rather than raising K further.

In [ ]:
n_replay = sum(subsets[f"{n}_replay"]["records"] for n in SOURCES)
n_deictic = subsets["deictic"]["records"]
K = mix.deictic_repeat(n_deictic, n_replay, REPLAY_PER_DEICTIC)
DATA_MIXTURE = mix.mixture({"deictic": K, **{f"{n}_replay": 1 for n in SOURCES}})
print(f"deictic {n_deictic:,} records x {K} = {n_deictic * K:,}; replay {n_replay:,}; "
      f"replay per deictic {n_replay / max(n_deictic * K, 1):.2f}")
print("--data_mixture", DATA_MIXTURE)
if K > 10:
    print(f"WARNING: each deictic record is seen {K} times per epoch: likely to overfit. Collect more episodes, "
          f"or lower REPLAY to about {round(n_deictic * 10 * REPLAY_PER_DEICTIC):,} records' worth of trajectories.")

### Checks: every record loads, every frame is on disk

In [ ]:
checks = {name: mix.check_dataset(s["data"], s["images"]) for name, s in subsets.items()}
for name, c in checks.items():
    print(f"{name:13} {c['records']:7,} records {c['frames']:8,} frames  missing {c['missing frames']:,}  "
          f"problems {c['problems'] or '-'}  {c['record share']}")
not_ready = [n for n, c in checks.items() if c["missing frames"] or c["problems"] or not c["records"]]
print("\nREADY" if not not_ready else f"\nNOT READY: {not_ready}")

### Registration and training script

`datasets_mixture_snippet.py` goes into `register_datasets_mixtures()` in NaVILA's
`llava/data/datasets_mixture.py`; `train_lora.sh` is NaVILA's `scripts/train/sft_8frames.sh` changed for
LoRA on the released model. Paths are as seen on the training machine (`TRAIN_DATA_DIR`).

In [ ]:
def on_trainer(p):
    p = Path(p)
    return str(Path(TRAIN_DATA_DIR) / p.relative_to(DATA)) if p.is_relative_to(DATA) else str(p)

snippet = mix.registration({name: (on_trainer(s["data"]), on_trainer(s["images"])) for name, s in subsets.items()})
(MIX_DIR / "datasets_mixture_snippet.py").write_text(snippet + "\n")
(MIX_DIR / "train_lora.sh").write_text(mix.train_script(DATA_MIXTURE))
(MIX_DIR / "mix_report.json").write_text(json.dumps(
    {"data_mixture": DATA_MIXTURE, "deictic_repeat": K, "replay_per_deictic": REPLAY_PER_DEICTIC,
     "target_stop_share": TARGET_STOP_SHARE, "replay": REPLAY, "eval": EVAL, "seed": SEED,
     "inference_settings": params, "deictic": deictic, "checks": checks, "ready": not not_ready}, indent=1, default=str))
print(snippet)
print("wrote", *(MIX_DIR / f for f in ("datasets_mixture_snippet.py", "train_lora.sh", "mix_report.json")), sep="\n  ")

### Samples as the model sees them

The prompt NaVILA builds and the 8 frames it samples from the history (black = padding, as in NaVILA's
loader). For our episodes the first decision should show the pointing person in the first frames.

In [ ]:
def show(name, i):
    s = subsets[name]
    recs = json.loads(Path(s["data"]).read_text())
    if not recs:
        print(name, "is empty")
        return
    r = recs[i % len(recs)]
    picked = mix.sample_history([Path(s["images"]) / f for f in r["frames"]])
    print(f"--- {name} {r['video_id']}: {len(r['frames'])} history frames -> answer: {r['a']}")
    print(mix.navila_prompt(r["q"]).replace("<image>\n", "<image> "))
    display(Image(data=mix.contact_sheet(picked)))

for name, i in (("deictic", 0), ("deictic", 1), ("r2r_replay", 7), ("rxr_replay", 7)):
    show(name, i)